In [36]:
!pip install -q langchain langchain-openai langgraph duckduckgo-search beautifulsoup4 requests


In [37]:
!pip install -q -U langchain langchain-openai ddgs beautifulsoup4 requests

import os
import requests
from bs4 import BeautifulSoup
from typing import List
from ddgs import DDGS
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent


In [47]:
!pip install -q -U langchain langchain-openai ddgs beautifulsoup4 requests

import os
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

os.environ["GROQ_API_KEY"] = "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"

model_groq = ChatOpenAI(
    model="llama-3.1-8b-instant",
    temperature=0,
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)

@tool
def internet_search(query: str) -> str:
    """Search the internet and return the top search results."""
    results_text = []

    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=5)

        for i, r in enumerate(results, start=1):
            title = r.get("title", "No title")
            url = r.get("href", "No URL")
            snippet = r.get("body", "No snippet")

            results_text.append(
                f"Result {i}\n"
                f"Title: {title}\n"
                f"URL: {url}\n"
                f"Snippet: {snippet}\n"
            )

    if not results_text:
        return "No search results found."

    return "\n".join(results_text)

@tool
def fetch_url(url: str) -> str:
    """Fetch readable text content from a URL."""
    try:
        response = requests.get(
            url,
            timeout=10,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)
        return text[:12000]

    except requests.exceptions.RequestException as e:
        return f"ERROR: Could not fetch {url}. Reason: {str(e)}"

system_prompt = """
You are a web research agent.

Your job is to answer the user's question using the content from the top-3 ranking websites.

Instructions:
1. Read the user's question carefully.
2. If useful, rewrite the question into a better and more search-friendly query before searching.
3. Call internet_search(query).
4. From the returned search results, select the top 3 most relevant results.
5. For each of those top 3 results, call fetch_url(url).
6. If one URL fails or returns an error, skip it and try another search result.
7. Read and compare the content from the successfully fetched websites.
8. Write one final answer using only the information from those fetched pages.
9. If the websites disagree, briefly mention the disagreement.
10. At the end, list the source URLs you used.
"""

agent = create_agent(
    model=model_groq,
    tools=[internet_search, fetch_url],
    system_prompt=system_prompt,
)

user_question = "What is the difference between RAG and fine-tuning in AI?"

response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": user_question}
        ]
    }
)

print(response["messages"][-1].content)

The difference between RAG and fine-tuning in AI is that RAG augments large language models (LLM) by connecting it to an organization's proprietary database, while fine-tuning optimizes models for domain-specific tasks. RAG avoids altering the model, while fine-tuning requires adjusting its parameters.

RAG and fine-tuning both aim to improve LLMs, but use different methods. RAG is ideal for fields where data is constantly changing, while fine-tuning is better suited for highly specialized tasks. The choice between RAG and fine-tuning depends on the specific use case and requirements.

Sources:

* https://www.ibm.com/think/topics/rag-vs-fine-tuning
* https://www.montecarlodata.com/blog-rag-vs-fine-tuning/
* https://www.redhat.com/en/topics/ai/rag-vs-fine-tuning
* https://aisera.com/blog/llm-fine-tuning-vs-rag/
* https://www.digitalocean.com/resources/articles/rag-vs-fine-tuning
